# 4xL4 NCCL P2P check (E025)

Topology, P2P status and NCCL all-reduce timing with default settings vs `NCCL_P2P_LEVEL=NODE/SYS`, plus a correctness check. No model is loaded.

In [ ]:
# Write the two scripts, then run topology checks and the all-reduce benches (no model, no vLLM).
import subprocess, os, textwrap
os.makedirs("/kaggle/working/p2p", exist_ok=True)
open("/kaggle/working/p2p/allreduce_bench.py", "w").write('"""NCCL all-reduce timing across the 4 GPUs for tensor sizes TP4 inference uses per all-reduce\n(bf16 [tokens, 5120]): decode steps (24-48 tokens) and prefill chunks (2048-8192 tokens).\nA TP4 forward of Qwen3.8-27B does 2 all-reduces per layer x 64 layers = 128 per step.\nusage: torchrun --nproc_per_node 4 allreduce_bench.py"""\nimport os, time, torch, torch.distributed as dist\n\ndist.init_process_group("nccl")\nr = dist.get_rank()\ntorch.cuda.set_device(r)\nfor tokens in (24, 48, 512, 2048, 4096, 8192):\n    x = torch.randn(tokens, 5120, dtype=torch.bfloat16, device="cuda")\n    for _ in range(5):\n        dist.all_reduce(x)\n    torch.cuda.synchronize()\n    n = 50 if tokens <= 2048 else 20\n    t0 = time.time()\n    for _ in range(n):\n        dist.all_reduce(x)\n    torch.cuda.synchronize()\n    dt = (time.time() - t0) / n\n    if r == 0:\n        mb = x.numel() * 2 / 1e6\n        print(f"tokens {tokens:5d}  {mb:7.1f} MB  {dt * 1e3:7.2f} ms per all-reduce  "\n              f"bus {2 * 3 / 4 * mb / 1e3 / dt:6.1f} GB/s  x128 per step = {128 * dt:6.3f} s", flush=True)\ndist.destroy_process_group()\n')
open("/kaggle/working/p2p/allreduce_check.py", "w").write('"""Correctness of NCCL all-reduce across the 4 GPUs (run with and without NCCL_P2P_LEVEL=SYS):\nrandom bf16/fp32 tensors of several sizes, all-reduce result compared with the sum computed from gathered inputs.\nusage: torchrun --nproc_per_node 4 allreduce_check.py"""\nimport torch, torch.distributed as dist\n\ndist.init_process_group("nccl")\nr = dist.get_rank()\nw = dist.get_world_size()\ntorch.cuda.set_device(r)\nbad = 0\nfor it in range(200):\n    for tokens in (1, 24, 48, 511, 2048, 8192):\n        g = torch.Generator(device="cuda").manual_seed(1000 * it + tokens * 7 + r)\n        x = torch.randn(tokens, 5120, dtype=torch.float32, device="cuda", generator=g)\n        parts = [torch.empty_like(x) for _ in range(w)]\n        dist.all_gather(parts, x)\n        ref = torch.stack(parts).sum(0)\n        y = x.clone()\n        dist.all_reduce(y)\n        err = (y - ref).abs().max().item()\n        if err > 1e-3:\n            bad += 1\n            if bad <= 5:\n                print(f"rank {r} it {it} tokens {tokens}: max abs err {err}", flush=True)\nt = torch.tensor([bad], device="cuda")\ndist.all_reduce(t)\nif r == 0:\n    print(f"all-reduce check: {int(t.item())} mismatching tensors out of {200 * 6 * w} rank-checks", flush=True)\ndist.destroy_process_group()\n')
def sh(cmd, env=None):
    print("$", cmd, flush=True)
    e = dict(os.environ); e.update(env or {})
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=e, timeout=900)
    print(r.stdout[-6000:], r.stderr[-3000:] if r.returncode else "", flush=True)
    return r.stdout + r.stderr
sh("nvidia-smi --query-gpu=name,pci.bus_id,driver_version --format=csv")
sh("nvidia-smi topo -m")
sh("nvidia-smi topo -p2p r")
sh("nvidia-smi topo -p2p w")
sh("env | grep -E '^(NCCL|CUDA_VISIBLE)' || true")
sh("python -c 'import torch; print(torch.__version__, torch.cuda.nccl.version())'")


In [ ]:
for name, env in [("default", {}), ("P2P_LEVEL=NODE", {"NCCL_P2P_LEVEL": "NODE"}), ("P2P_LEVEL=SYS", {"NCCL_P2P_LEVEL": "SYS"})]:
    print("=" * 20, name, flush=True)
    out = sh("NCCL_DEBUG=INFO torchrun --nproc_per_node 4 /kaggle/working/p2p/allreduce_bench.py 2>&1 | grep -E '^tokens|via (P2P|SHM|NET)' | sort | uniq -c | sort -rn | head -12", env)
print("=" * 20, "correctness, P2P_LEVEL=SYS", flush=True)
sh("torchrun --nproc_per_node 4 /kaggle/working/p2p/allreduce_check.py 2>&1 | grep -E 'all-reduce check|err' | head -6", {"NCCL_P2P_LEVEL": "SYS"})
